# 01. Adquisicion y preparacion de datos

## Situacion problematica

ACRE Africa ofrece seguros agricolas a pequenos productores de Africa oriental. Cuando un
agricultor presenta una reclamacion, un evaluador revisa las fotografias que el propio
agricultor toma con su telefono, decide si el cultivo esta danado, identifica la causa y estima
el porcentaje de perdida; de esa estimacion depende el monto que se paga. Esa revision manual es
el cuello de botella del proceso y la sequia concentra la mayoria de las reclamaciones.

El proyecto Eyes on the Ground construyo un conjunto de datos etiquetado para entrenar modelos
que sustituyan esa revision. Los modelos ajustados sobre las primeras temporadas alcanzaron un
desempeno aceptable que no se sostuvo al aplicarlos a una temporada posterior.

## Problema cientifico

Identificar que caracteristicas del conjunto de datos limitan la generalizacion entre temporadas
y que preprocesamiento permite estimar de forma confiable la magnitud del dano por sequia a
partir de fotografias.

## Objetivos especificos

1. Describir la estructura y la calidad de las variables tabulares y de las imagenes,
   cuantificando valores faltantes, duplicados e inconsistencias entre el tipo de dano declarado
   y su magnitud.
2. Comparar la distribucion de las variables clave entre temporadas y entre los conjuntos de
   entrenamiento y prueba, midiendo la magnitud del desplazamiento de dominio.
3. Cuantificar la agrupacion de imagenes por campo y su efecto sobre el riesgo de fuga de
   informacion al particionar los datos.

Este primer cuaderno documenta la obtencion reproducible del conjunto, describe sus variables y
deja registradas las decisiones de limpieza antes del analisis exploratorio.

## Preparacion del entorno

La primera celda localiza la raiz del proyecto subiendo desde el directorio de trabajo hasta el
primer directorio que contiene una carpeta `src`, y la agrega a `sys.path`. Asi el cuaderno
funciona sin importar desde donde se inicie Jupyter.

In [1]:
import sys
from pathlib import Path


def raiz_proyecto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "src").is_dir():
            return candidato
    raise RuntimeError("No se encontro la raiz del proyecto")


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src import carga, descarga, limpieza, tablas
from src.config import CONTEO_OFICIAL_TEMPORADA, ETAPAS_CRECIMIENTO, TIPOS_DANO

## Obtencion reproducible de los datos

Los datos se descargan con `src.descarga`, que trae los CSV desde la API de Zindi y las imagenes
desde Google Drive; las instrucciones completas estan en el `README.md`. La verificacion siguiente
confirma el estado local de los archivos sin volver a descargar nada.

In [2]:
descarga.imprimir_verificacion(descarga.verificar())

Archivos CSV
  Train.csv: 1489.2 KB
  Test.csv: 476.5 KB
  SampleSubmission.csv: 135.4 KB
Imagenes por particion
  train: 26068
  test: 8663
Imagenes de entrenamiento por temporada
  LR2020: 2033 de 2034 -> difiere en -1
  LR2021: 7945 de 7979 -> difiere en -34
  SR2020: 6163 de 6163 -> coincide
  SR2021: 9927 de 9930 -> difiere en -3
  total: 26068 de 26106
Correspondencia entre Train.csv e imagenes en disco
  registros en el csv: 26068
  registros sin imagen en disco: 0
  imagenes en disco sin registro: 0


El conteo local de imagenes por temporada queda por debajo del conteo oficial documentado por la
competencia. La siguiente tabla cuantifica la diferencia temporada por temporada.

In [3]:
train = carga.cargar_train()
test = carga.cargar_test()

conteo_local = train["temporada"].value_counts().reindex(CONTEO_OFICIAL_TEMPORADA.keys())
comparacion_temporada = pd.DataFrame(
    {"oficial": pd.Series(CONTEO_OFICIAL_TEMPORADA), "local": conteo_local}
)
comparacion_temporada["diferencia"] = comparacion_temporada["local"] - comparacion_temporada["oficial"]
comparacion_temporada

,oficial,local,diferencia
LR2020,2034,2033,-1
LR2021,7979,7945,-34
SR2020,6163,6163,0
SR2021,9930,9927,-3


La correspondencia entre `Train.csv` y las imagenes en disco es exacta: no hay registros sin
imagen ni imagenes sin registro. Por lo tanto la diferencia frente al conteo oficial no proviene
de una descarga incompleta, sino de que el conjunto distribuido contiene unos pocos registros
menos que la cifra documentada. Se trabaja con el conjunto tal como se distribuye y se deja
constancia de la diferencia.

## Lectura de los conjuntos y variables derivadas

Las columnas originales del CSV se renombran al espanol: `ID` a identificador, `filename` a
archivo, `growth_stage` a etapa, `damage` a dano, `extent` a magnitud y `season` a temporada. El
nombre de archivo no es arbitrario sino un identificador estructurado con dos convenciones: un
formato codificado (`L<productor>F<campo>C<cultivo>S<sitio><tipo><secuencia>.jpg`) usado por tres
temporadas, y un formato secuencial (`<productor>_<tipo>_<n>_<resto>.JPG`) usado por la temporada
mas antigua. De ambos se derivan el productor, el campo, el cultivo, el sitio y el tipo de
captura. El identificador de campo es clave porque las fotografias de un mismo campo son
observaciones dependientes, y repartirlas al azar entre entrenamiento y validacion filtraria
informacion.

In [4]:
print(f"train: {train.shape[0]} registros, {train.shape[1]} columnas")
print(f"test:  {test.shape[0]} registros, {test.shape[1]} columnas")
train.head()

train: 26068 registros, 17 columnas
test:  8663 registros, 16 columnas


,id,archivo,etapa,dano,magnitud,temporada,particion,formato_nombre,id_productor,id_campo,codigo_cultivo,id_sitio,tipo_captura,es_copia,es_repeticion,ruta_relativa,existe_archivo
0,ID_1S8OOWQYCB,L427F01330C01S03961Rp02052.jpg,S,WD,0,SR2020,train,codificado,L427,L427F01330,C01,S03961,seguimiento,False,False,train/L427F01330C01S03961Rp02052.jpg,True
1,ID_0MD959MIZ0,L1083F00930C39S12674Ip.jpg,V,G,0,SR2021,train,desconocido,NaN,NaN,NaN,NaN,NaN,False,False,train/L1083F00930C39S12674Ip.jpg,True
2,ID_JRJCI4Q11V,24_initial_1_1463_1463.JPG,V,G,0,LR2020,train,secuencial,U24,U24,NaN,NaN,inicial,False,False,train/24_initial_1_1463_1463.JPG,True
3,ID_DBO3ZGI1GM,L341F00167C01S00324Rp14178.jpg,M,DR,60,SR2020,train,codificado,L341,L341F00167,C01,S00324,seguimiento,False,False,train/L341F00167C01S00324Rp14178.jpg,True
4,ID_ORZLWTEUUS,L1084F02394C39S13931Ip.jpg,V,G,0,SR2021,train,desconocido,NaN,NaN,NaN,NaN,NaN,False,False,train/L1084F02394C39S13931Ip.jpg,True


`Test.csv` no incluye la columna de magnitud porque ese es precisamente el valor que la
competencia pide estimar; si trae el tipo de dano declarado, lo que sera relevante para acotar el
problema mas adelante.

## Diccionario de datos

El diccionario resume, para cada variable, su tipo, su escala de medicion, la cantidad de valores
presentes y ausentes, el numero de valores distintos y un ejemplo.

In [5]:
carga.diccionario_datos(train)

,variable,tipo,escala,no_nulos,nulos,unicos,ejemplo
0,id,str,texto,26068,0,26068,ID_1S8OOWQYCB
1,archivo,str,texto,26068,0,26068,L427F01330C01S03961Rp02052.jpg
2,etapa,category,ordinal,26068,0,4,S
3,dano,category,nominal,26068,0,8,WD
4,magnitud,int64,cuantitativa,26068,0,11,0
5,temporada,category,ordinal,26068,0,4,SR2020
6,particion,str,texto,26068,0,1,train
7,formato_nombre,category,nominal,26068,0,3,codificado
8,id_productor,str,texto,23356,2712,221,L427
9,id_campo,str,texto,23356,2712,3051,L427F01330


Los codigos de etapa de crecimiento y de tipo de dano son abreviaturas de dos y tres letras. Los
catalogos de referencia permiten interpretarlas.

In [6]:
catalogo_etapa = pd.DataFrame(ETAPAS_CRECIMIENTO.items(), columns=["codigo", "descripcion"])
catalogo_dano = pd.DataFrame(TIPOS_DANO.items(), columns=["codigo", "descripcion"])
display(catalogo_etapa)
display(catalogo_dano)

,codigo,descripcion
0,S,Siembra
1,V,Vegetativa
2,F,Floracion
3,M,Madurez


,codigo,descripcion
0,DR,Sequia
1,DS,Enfermedad
2,FD,Inundacion
3,G,Crecimiento sano
4,ND,Deficiencia de nutrientes
5,PS,Plaga
6,WD,Maleza
7,WN,Viento


## Limpieza y preprocesamiento

El enfoque es centrado en datos: las operaciones de limpieza se aplican solo cuando hay una razon
documentada. No se imputan valores por el simple hecho de que falten, porque en un conjunto de
etiquetas humanas la ausencia de un valor tambien es informacion sobre el proceso de anotacion.
En lugar de eliminar registros, se agregan indicadores de calidad que permiten filtrar segun el
proposito de cada analisis posterior.

### Valores faltantes

In [7]:
limpieza.resumen_faltantes(train)

,variable,faltantes,porcentaje
0,id_sitio,4745,18.202
1,codigo_cultivo,4745,18.202
2,tipo_captura,2712,10.404
3,id_productor,2712,10.404
4,id_campo,2712,10.404
5,id,0,0.000
6,archivo,0,0.000
7,dano,0,0.000
8,etapa,0,0.000
9,formato_nombre,0,0.000


### Duplicados y composicion por tipo de captura

Ademas de revisar duplicados de identificador y de nombre de archivo, se cruzan la temporada con
el formato de nombre y con el tipo de captura. Una composicion desigual del tipo de captura entre
temporadas afecta la comparabilidad, porque cada tipo de captura tiene un proposito distinto y,
como se vera, una magnitud tipica distinta.

In [8]:
limpieza.duplicados_registro(train)

,variable,registros_duplicados,valores_afectados
0,id,0,0
1,archivo,0,0


In [9]:
tablas.tabla_contingencia(train, "temporada", "formato_nombre")

formato_nombre,codificado,desconocido,secuencial
temporada,,,
LR2020,0,0,2033
SR2020,5347,816,0
LR2021,7945,0,0
SR2021,8031,1896,0


In [10]:
tablas.tabla_contingencia(train, "temporada", "tipo_captura")

tipo_captura,inicial,seguimiento,reclamo
temporada,,,
LR2020,153,1880,0
SR2020,0,5347,0
LR2021,0,7945,0
SR2021,0,7572,459


### Validacion de dominios

Se comprueba que cada variable respete su dominio esperado: la etapa dentro del catalogo de
cuatro valores, el tipo de dano dentro del catalogo de ocho valores, y la magnitud entre cero y
cien y multiplo de diez.

In [11]:
limpieza.validar_dominios(train)

,variable,regla,violaciones
0,etapa,"valor en ['S', 'V', 'F', 'M']",0
1,dano,"valor en ['G', 'DR', 'DS', 'FD', 'ND', 'PS', '...",0
2,magnitud,rango 0 a 100,0
3,magnitud,multiplo de 10,0


### Regla estructural entre tipo de dano y magnitud

La magnitud solo deberia tomar valores positivos cuando el dano declarado es sequia. Esta
verificacion documenta el nivel de ruido de etiquetado y condiciona el desempeno maximo
alcanzable por cualquier modelo. Los casos que la rompen no se corrigen: se conservan y se
analizan como ruido.

In [12]:
limpieza.regla_estructural(train)

,caso,registros,porcentaje
0,magnitud positiva con dano distinto de sequia,0,0.000
1,sequia declarada con magnitud cero,6,0.023
2,crecimiento sano con magnitud positiva,0,0.000


### Correspondencia entre registros e imagenes

El analisis posterior combina la tabla de etiquetas con la tabla de atributos de imagen, de modo
que ambos conjuntos deben coincidir. Se cuentan los registros sin imagen en disco y las imagenes
sin registro asociado, para entrenamiento y para prueba.

In [13]:
for particion, df in (("train", train), ("test", test)):
    sin_imagen = limpieza.registros_sin_imagen(df)
    huerfanas = limpieza.imagenes_huerfanas(df, particion)
    print(f"{particion}: {len(sin_imagen)} registros sin imagen, {len(huerfanas)} imagenes sin registro")

train: 0 registros sin imagen, 0 imagenes sin registro
test: 0 registros sin imagen, 0 imagenes sin registro


### Bitacora de decisiones

La bitacora consolida todas las verificaciones anteriores junto con la decision tomada en cada
caso, dejando explicito que ninguna elimina informacion.

In [14]:
limpieza.bitacora(train, "train")

,verificacion,registros,decision
0,registros leidos,26068,sin accion
1,valores faltantes en variables clave,0,"no se imputa, se conserva el faltante y se doc..."
2,identificadores duplicados,0,se revisa el origen antes de eliminar
3,formato de nombre desconocido,2712,se revisa el patron antes de derivar variables...
4,tipo de captura no identificado,2712,se conservan y se excluyen de los cruces que u...
5,registros sin imagen en disco,0,se excluyen del modelado
6,imagenes en disco sin registro,0,se excluyen del analisis tabular
7,temporada no identificada,0,se revisa el patron del nombre de archivo
8,campos unicos identificados,3051,definen los grupos de particion
9,magnitud fuera de dominio,0,se excluyen del modelado


### Marcado de calidad y guardado

Se agregan a cada registro los indicadores `magnitud_valida`, `etiqueta_coherente` y
`apto_modelado`, y se guardan los metadatos de entrenamiento y prueba en formato Parquet. El
cuaderno siguiente parte de estos metadatos y no vuelve a leer los CSV originales.

In [15]:
train_marcado = limpieza.marcar_calidad(train)
test_marcado = limpieza.marcar_calidad(test)

ruta_train = carga.guardar_metadatos(train_marcado, "train")
ruta_test = carga.guardar_metadatos(test_marcado, "test")

aptos = int(train_marcado["apto_modelado"].sum())
print(f"registros aptos para modelado: {aptos} de {len(train_marcado)}")
print(f"metadatos guardados en:")
print(f"  {ruta_train}")
print(f"  {ruta_test}")

registros aptos para modelado: 23356 de 26068
metadatos guardados en:
  /home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/CC3084-Proyecto2/data/procesado/metadatos_train.parquet
  /home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/CC3084-Proyecto2/data/procesado/metadatos_test.parquet


## Cierre de la etapa

El conjunto quedo descrito, verificado contra su fuente original y marcado con indicadores de
calidad, sin eliminar informacion. El cuaderno `02_eda_tabular` retoma los metadatos guardados
para estudiar las etiquetas: la magnitud del dano, las variables categoricas, sus relaciones, y
los dos riesgos que condicionan cualquier modelo posterior, la agrupacion por campo y el
desplazamiento entre temporadas.